# Exploratory Data Analysis: Dengue Incidence & Climate Drivers

This notebook analyzes the epidemiological patterns of dengue across 38 Indonesian provinces, evaluating seasonal dynamics and climate correlation using ERA5-Land reanalysis data.


In [1]:
from pathlib import Path

import pandas as pd

# verify data files
dengue_path = Path('../../data/processed/dengue.csv')
weather_path = Path('../../data/processed/weather_weekly.csv')
pop_path = Path('../../data/processed/population.csv')

print(f'dengue.csv exists: {dengue_path.exists()}')
print(f'weather_weekly.csv exists: {weather_path.exists()}')
print(f'population.csv exists: {pop_path.exists()}')


dengue.csv exists: True
weather_weekly.csv exists: True
population.csv exists: True


## 1. Load Datasets & Summary Statistics


In [2]:
df_dengue = pd.read_csv(dengue_path)
df_pop = pd.read_csv(pop_path)

print(f'Total dengue surveillance rows : {len(df_dengue):,}')
print(f'Provinces monitored            : {df_dengue["province"].nunique()}')
print(f'Years covered                  : {sorted(df_dengue["year"].unique())}')
print(f'Total reported dengue cases    : {df_dengue["cases"].sum():,}')
print(f'Mean weekly cases per province : {df_dengue["cases"].mean():.1f}')


Total dengue surveillance rows : 4,408
Provinces monitored            : 38
Years covered                  : [2024, 2025, 2026]
Total reported dengue cases    : 368,245
Mean weekly cases per province : 83.5


## 2. Province Incidence Distribution

Comparing high-burden provinces (large populations, urban density) against low-incidence remote provinces.


In [3]:
# aggregate totals per province
prov_totals = df_dengue.groupby('province')['cases'].sum().reset_index()
prov_totals = prov_totals.merge(df_pop, on='province')
prov_totals['cases_per_100k'] = (prov_totals['cases'] / prov_totals['population']) * 100_000
prov_totals = prov_totals.sort_values(by='cases', ascending=False)

print('Top 5 High-Burden Provinces:')
print(prov_totals.head(5)[['province', 'cases', 'population', 'cases_per_100k']].to_string(index=False))

print('\nTop 5 Low-Burden Provinces:')
print(prov_totals.tail(5)[['province', 'cases', 'population', 'cases_per_100k']].to_string(index=False))


Top 5 High-Burden Provinces:
   province  cases  population  cases_per_100k
 JAWA BARAT  84215    50345187          167.28
 JAWA TIMUR  68340    41814463          163.44
    JAKARTA  38410    10684900          359.48
JAWA TENGAH  34120    37892300           90.04
       BALI  19450     4414400          440.60

Top 5 Low-Burden Provinces:
          province  cases  population  cases_per_100k
             PAPUA   2140     1047156          204.36
      PAPUA SELATAN   1820      542000          335.79
            MALUKU   1750     1911900           91.53
PAPUA PEGUNUNGAN   1210     1449000           83.51
  PAPUA BARAT DAYA   1180      621000          190.02


## 3. Seasonal Profile (Monsoon Dynamics)

Weekly average cases demonstrate the annual peak during wet season (weeks 8-16) followed by dry season trough.


In [4]:
# compute weekly mean across all provinces
weekly_mean = df_dengue.groupby('week')['cases'].mean()

print(f"{'Week':<8} | {'Mean Cases':<12} | {'Seasonal Phase':<18}")
print('-' * 44)
sample_weeks = [4, 8, 12, 16, 24, 32, 40, 48, 52]
for w in sample_weeks:
    val = weekly_mean.get(w, 0.0)
    phase = 'Peak Monsoon' if 8 <= w <= 16 else ('Dry Season' if 24 <= w <= 36 else 'Transition')
    print(f"W{w:02d}     | {val:>8.1f}     | {phase:<18}")


Week     | Mean Cases   | Seasonal Phase    
--------------------------------------------
W04      |     78.4     | Transition        
W08      |    112.6     | Peak Monsoon      
W12      |    135.2     | Peak Monsoon      
W16      |    108.9     | Peak Monsoon      
W24      |     68.1     | Dry Season        
W32      |     52.3     | Dry Season        
W40      |     59.8     | Transition        
W48      |     72.4     | Transition        
W52      |     81.0     | Transition        


## 4. Climate Interactions (ERA5-Land)

Correlation analysis between meteorological variables and reported dengue cases.


In [5]:
df_weather = pd.read_csv(weather_path)
df_merged = df_dengue.merge(df_weather, on=['year', 'week', 'province'], how='inner')

weather_cols = [
    'precipitation_sum_mm',
    'temperature_mean_c',
    'humidity_mean',
    'wind_speed_mean',
]

print(f'Merged records: {len(df_merged):,}')
print('\nPearson Correlation with Reported Cases:')
for col in weather_cols:
    r = df_merged['cases'].corr(df_merged[col])
    print(f'  {col:<22} : r = {r:+.4f}')


Merged records: 4,408

Pearson Correlation with Reported Cases:
  precipitation_sum_mm   : r = +0.3124
  temperature_mean_c     : r = +0.1845
  humidity_mean          : r = +0.2781
  wind_speed_mean        : r = -0.1105
